# Transcription Factor Binding Prediction with OmniGenBench

This notebook provides a step-by-step demonstration to extend OmniGenBench to the TFB task based on the **OmniGenome-52M** model on the **DeepSEA dataset**. The goal is to perform multi-label classification to predict the binding sites of various transcription factors based on DNA sequences.

**Dataset Description:**
The dataset used in this notebook is derived from the DeepSEA dataset, which is designed for studying the effects of non-coding variants. It consists of DNA sequences of 1000 base pairs, each associated with 919 binary labels corresponding to various chromatin features (transcription factor binding, DNase I sensitivity, and histone marks). For this task, we use a preprocessed version available from the [`deepsea_tfb_prediction`](https://huggingface.co/datasets/yangheng/tfb_prediction) dataset on Hugging Face.

**Estimated Runtime:**
The total runtime for this notebook depends on the hardware and the number of training examples (`MAX_EXAMPLES`). On a single NVIDIA RTX 4090 GPU, training with the default settings (`MAX_EXAMPLES=100000`, `EPOCHS=10`) takes approximately **1–2 hours**. For a quick test run with `MAX_EXAMPLES=1000`, it should take about **5–10 minutes**.


## Notebook Structure

This notebook is organized into concise sections. Most core logic is moved to [`examples/tfb_prediction/utils.py`](https://github.com/COLA-Laboratory/OmniGenBench/blob/master/examples/tfb_prediction/utils.py) and imported here:

1. **Setup & Installation**: Ensures all required libraries and dependencies are installed.
2. **Import Libraries**: Loads the necessary Python libraries for genomic data processing, model inference, and analysis.
3. **Configuration**: Defines key parameters such as file paths, model selection, and training hyperparameters.
4. **Model and Dataset Initialization**: Initializes the tokenizer, model, datasets, and data loaders.
5. **Finetuning**: Fine-tunes the model using `AccelerateTrainer` via utility functions.
6. **Inference Example**: Uses the trained model to make predictions on a new DNA sequence.

Follow the notebook sequentially to execute the TFB prediction pipeline effectively.

## 1. Setup & Installation

First, let's ensure all the required packages are installed. If you have already installed them, you can skip this cell. Otherwise, uncomment and run the cell to install the dependencies.

In [ ]:
!pip install -U numpy transformers omnigenbench autocuda

## 2. Import Libraries

Import all the necessary libraries for genomic data processing, model inference, and analysis.

In [ ]:
import autocuda
import findfile
import zipfile
import os
import json
import requests
import torch
import torch.nn as nn
from typing import Optional, List, Dict, Any
from transformers import AutoTokenizer, AutoModel

from omnigenbench import (
    OmniDataset,
    ClassificationMetric,
    AccelerateTrainer,
    ModelHub,
    OmniModelForMultiLabelSequenceClassification,
    OmniModel,
    # Integrated baselines available for completeness
    OmniCNNBaseline,
    OmniRNNBaseline,
    OmniBasenjiBaseline,
    OmniBPNetBaseline,
    OmniDeepSTARRBaseline,
)


class _TokenIdsToOneHot4(nn.Module):
    """Map tokenizer input_ids [B,L] to one-hot channels [B,4,L] for A,C,G,T.

    Unknown tokens (including N and pads) map to all-zero columns.
    """
    def __init__(self, tokenizer):
        super().__init__()
        vocab = tokenizer.get_vocab() if hasattr(tokenizer, "get_vocab") else {}
        token_to_channel = {"A": 0, "C": 1, "G": 2, "T": 3, "a": 0, "c": 1, "g": 2, "t": 3}
        vocab_size = len(vocab) if vocab else getattr(tokenizer, "vocab_size", 0) or 0
        mapping = torch.full((max(vocab_size, 1),), fill_value=-1, dtype=torch.long)
        if vocab:
            for tok, idx in vocab.items():
                if tok in token_to_channel:
                    mapping[idx] = token_to_channel[tok]
        else:
            for ch, ch_idx in token_to_channel.items():
                try:
                    idx = tokenizer.convert_tokens_to_ids(ch)
                    if isinstance(idx, int) and idx >= 0:
                        if idx >= mapping.numel():
                            new_map = torch.full((idx + 1,), fill_value=-1, dtype=torch.long)
                            new_map[: mapping.numel()] = mapping
                            mapping = new_map
                        mapping[idx] = ch_idx
                except Exception:
                    pass
        self.register_buffer("id2chan", mapping)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        if input_ids.dtype != torch.long:
            input_ids = input_ids.long()
        vocab_to_channel = self.id2chan
        max_id = int(input_ids.max().item()) if input_ids.numel() > 0 else -1
        if max_id >= vocab_to_channel.numel():
            new_map = torch.full((max_id + 1,), fill_value=-1, dtype=torch.long, device=vocab_to_channel.device)
            new_map[: vocab_to_channel.numel()] = vocab_to_channel
            self.id2chan = new_map  # type: ignore[attr-defined]
            vocab_to_channel = self.id2chan
        idx = vocab_to_channel[input_ids]
        idx_clamped = idx.clamp(min=-1)
        idx_clamped = torch.where(idx_clamped < 0, torch.full_like(idx_clamped, 4), idx_clamped)
        one_hot5 = torch.nn.functional.one_hot(idx_clamped, num_classes=5).to(torch.float32)
        x = one_hot5[..., :4].permute(0, 2, 1).contiguous()
        return x


def download_deepsea_dataset(local_dir):
    if not findfile.find_cwd_dir(local_dir, disable_alert=True):
        os.makedirs(local_dir, exist_ok=True)
    url_to_download = "https://huggingface.co/datasets/yangheng/deepsea_tfb_prediction/resolve/main/deepsea_tfb_prediction.zip"
    zip_path = os.path.join(local_dir, "deepsea_tfb_prediction.zip")
    if not os.path.exists(zip_path):
        print(f"Downloading deepsea_tfb_prediction.zip from {url_to_download}...")
        response = requests.get(url_to_download, stream=True)
        response.raise_for_status()
        with open(zip_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Downloaded {zip_path}")

    ZIP_DATASET = findfile.find_cwd_file("deepsea_tfb_prediction.zip")
    if ZIP_DATASET:
        with zipfile.ZipFile(ZIP_DATASET, 'r') as zip_ref:
            zip_ref.extractall(local_dir)
        print(f"Extracted deepsea_tfb_prediction.zip into {local_dir}")
        os.remove(ZIP_DATASET)
    else:
        print("deepsea_tfb_prediction.zip not found. Skipping extraction.")


class DeepSEADataset(OmniDataset):
    """
    A dataset for the DeepSEA task that converts DNA sequences to tokenized inputs.
    Accepts JSONL where each line contains a `sequence` field and `label(s)`.
    """
    def __init__(
        self,
        data_source: str,
        tokenizer,
        max_length: Optional[int] = None,
        **kwargs,
    ) -> None:
        super().__init__(data_source, tokenizer, max_length, **kwargs)
        self.label_indices = None
        for key, value in kwargs.items():
            self.metadata[key] = value

    def prepare_input(self, instance: Dict[str, Any], **kwargs) -> Dict[str, torch.Tensor]:
        def truncate(seq: str, windowsize: Optional[int]) -> str:
            if windowsize is None:
                return seq
            if len(seq) == windowsize:
                return seq
            if len(seq) > windowsize:
                left = (len(seq) - windowsize) // 2
                return seq[left:left + windowsize]
            return seq + ("N" * (windowsize - len(seq)))

        sequence = instance.get('sequence') or instance.get('seq')
        labels = instance.get('label', None) if 'label' in instance else instance.get('labels', None)

        tokenized_inputs = self.tokenizer(
            truncate(sequence, self.max_length),
            padding="do_not_pad",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
            **kwargs,
        )

        labels_tensor = torch.tensor(labels, dtype=torch.float32) if labels is not None else None
        if labels_tensor is not None and hasattr(self, 'label_indices') and self.label_indices is not None:
            labels_tensor = labels_tensor[torch.tensor(self.label_indices, dtype=torch.long)]
        tokenized_inputs["labels"] = labels_tensor

        for col in tokenized_inputs:
            if isinstance(tokenized_inputs[col], torch.Tensor) and tokenized_inputs[col].ndim > 1:
                tokenized_inputs[col] = tokenized_inputs[col].squeeze(0)

        return tokenized_inputs


def load_tokenizer_and_model(
    model_name_or_path: str,
    num_labels: int,
    threshold: float = 0.5,
    device: Optional[torch.device] = None,
):
    """
    Load tokenizer and OmniGenome-based multi-label classification model.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
    base_model = AutoModel.from_pretrained(model_name_or_path, trust_remote_code=True)

    model = OmniModelForMultiLabelSequenceClassification(
        base_model,
        tokenizer,
        num_labels=num_labels,
        threshold=threshold,
    )
    if device is not None:
        model = model.to(device).to(torch.float32)

    return tokenizer, model


def build_datasets(
    tokenizer,
    train_file: str,
    test_file: str,
    valid_file: Optional[str],
    max_length: int,
    max_examples: Optional[int],
    label_indices: Optional[List[int]] = None,
):
    """
    Create DeepSEADataset instances for train/valid/test.
    """
    def make_ds(path: str) -> DeepSEADataset:
        ds = DeepSEADataset(
            data_source=path,
            tokenizer=tokenizer,
            max_length=max_length,
            max_examples=max_examples,
            force_padding=False,
        )
        ds.label_indices = label_indices
        return ds

    train_set = make_ds(train_file)
    test_set = make_ds(test_file)
    valid_set = make_ds(valid_file) if (valid_file and os.path.exists(valid_file)) else None
    return train_set, valid_set, test_set


def create_dataloaders(
    train_set: torch.utils.data.Dataset,
    valid_set: Optional[torch.utils.data.Dataset],
    test_set: Optional[torch.utils.data.Dataset],
    batch_size: int,
):
    train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    test_loader = torch.utils.data.DataLoader(test_set, batch_size=batch_size) if test_set is not None else None
    valid_loader = torch.utils.data.DataLoader(valid_set, batch_size=batch_size) if valid_set is not None else None
    return train_loader, valid_loader, test_loader


def run_finetuning(
    model: torch.nn.Module,
    train_loader,
    valid_loader,
    test_loader,
    epochs: int,
    learning_rate: float,
    weight_decay: float,
    patience: int,
    device: torch.device,
    save_dir: str = "tfb_model",
    seed: Optional[int] = 2024,
):
    """
    Train the model with AccelerateTrainer and save to `save_dir`.
    Returns the trainer and the best metrics.
    """
    metric_fn = [ClassificationMetric(ignore_y=-100).roc_auc_score]
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    trainer = AccelerateTrainer(
        model=model,
        train_loader=train_loader,
        eval_loader=valid_loader,
        test_loader=test_loader,
        optimizer=optimizer,
        epochs=epochs,
        compute_metrics=metric_fn,
        patience=patience,
        device=device,
        seed=seed,
    )

    metrics_best = None
    if not os.path.exists(save_dir):
        metrics_best = trainer.train()
        trainer.save_model(save_dir)
    else:
        metrics_best = {"info": f"Found existing '{save_dir}'. Skipped training."}

    return trainer, metrics_best


def run_inference(
    model_dir: str,
    tokenizer,
    sample_sequence: str,
    max_length: int,
    device: torch.device,
):
    """
    Load a saved model via ModelHub and run a single-sequence inference.
    Returns a dict containing predictions and probabilities when available.
    """
    model = ModelHub.load(model_dir)
    model.to(device)

    inputs = tokenizer(sample_sequence, return_tensors="pt", max_length=max_length, truncation=True)
    inputs = inputs.to(torch.float32)
    with torch.no_grad():
        outputs = model.inference(inputs, device=device)
    return outputs


class _MaskedGlobalMaxPool1d(nn.Module):
    def forward(self, x: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        if attention_mask is None:
            return x.max(dim=1).values
        masked_x = x.masked_fill(attention_mask.unsqueeze(-1).eq(0), float("-inf"))
        return masked_x.max(dim=1).values


print("Libraries imported successfully.")


## 3. Configuration

Here, we define all the hyperparameters and settings for our experiment. This centralized configuration makes it easy to modify parameters and track experiments.

In [ ]:
# --- Data File Paths ---
LOCAL_PATH = "deepsea_tfb_prediction"
download_deepsea_dataset(LOCAL_PATH)
TRAIN_FILE = findfile.find_cwd_file(['train', 'jsonl'])
TEST_FILE =  findfile.find_cwd_file(['test', 'jsonl'])
VALID_FILE =  findfile.find_cwd_file(['valid', 'jsonl'])

# --- Available Models for Testing ---
AVAILABLE_MODELS = [
    'yangheng/OmniGenome-52M',
    'yangheng/OmniGenome-186M',
    'yangheng/OmniGenome-v1.5',
    # You can add more models here as needed,
]

MODEL_NAME_OR_PATH = AVAILABLE_MODELS[0]
USE_CONV_LAYERS = False  # Set to True to add DeepSEA-style convolutional layers on top of OmniGenome (not used in this demo)

# --- Training Hyperparameters ---
EPOCHS = 50
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 1e-3
BATCH_SIZE = 16
PATIENCE = 3  # For early stopping
MAX_LENGTH = 200  # The length of the DNA sequence to be processed
SEED = 45
# LABEL_INDICES = [0]  # Example indices for the first 10 transcription factors
LABEL_INDICES = list(range(919))
MAX_EXAMPLES = 1000  # Use a smaller number for quick testing (e.g., 1000), or None for all data

DEVICE = autocuda.auto_cuda()
print(f"Using device: {DEVICE}")


## 4. Model and Dataset Initialization

Initialize tokenizer and model, then build datasets and dataloaders using utilities for a concise workflow.

In [ ]:

# 1. Initialize Tokenizer and Model
print("--- Initializing Tokenizer and Model ---")

# Use utility to load tokenizer and model
label_count = len(LABEL_INDICES)
tokenizer, model = load_tokenizer_and_model(
    MODEL_NAME_OR_PATH,
    num_labels=label_count,
    threshold=0.5,
    device=DEVICE,
)

# 2. Create Datasets via utility
print("\n--- Creating Datasets ---")
train_set, valid_set, test_set = build_datasets(
    tokenizer=tokenizer,
    train_file=TRAIN_FILE,
    test_file=TEST_FILE,
    valid_file=VALID_FILE,
    max_length=MAX_LENGTH,
    max_examples=MAX_EXAMPLES,
    label_indices=LABEL_INDICES,
)

# Create DataLoaders for batching (utils)
train_loader, valid_loader, test_loader = create_dataloaders(
    train_set=train_set,
    valid_set=valid_set,
    test_set=test_set,
    batch_size=BATCH_SIZE,
)

print("\n--- Initialization Complete ---")
print(f"Training set size: {len(train_set)}")
print(f"Test set size: {len(test_set)}")
if valid_set:
    print(f"Validation set size: {len(valid_set)}")


## 5. Finetuning

Fine-tune the model using `AccelerateTrainer` (invoked through the `run_finetuning` compatibility wrapper). Early stopping monitors validation ROC AUC when a validation set is provided.

In [ ]:


# Train with utilities
print("--- Starting Training ---")
trainer, metrics_best = run_finetuning(
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    test_loader=test_loader,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    patience=PATIENCE,
    device=DEVICE,
    save_dir="tfb_model",
)
print(metrics_best)
print("--- Training Finished ---")


## 6. Deployment and Inference

Run a single-sequence prediction using the persisted fine-tuned model. The same preprocessing pathway (`encode_tokens`) ensures parity with training.

In [ ]:

from omnigenbench import OmniTokenizer
sample_sequence = "AGCT" * (512 // 4)  # Construct sequence of required length
tokenizer = OmniTokenizer.from_pretrained('yangheng/OmniGenome-52M')
outputs = run_inference(
    model_dir="yangheng/ogb_tfb_finetuned",
    tokenizer=tokenizer,
    sample_sequence=sample_sequence,
    max_length=len(sample_sequence),
    device='cuda',
)


## Inference Parsing

In [ ]:

predictions = outputs.get('predictions', None)
probabilities = outputs.get('probabilities', None)

print(f"Input sequence length: {len(sample_sequence)} bp")
if predictions is not None:
    print(f"Number of predicted labels: {len(predictions)}")
    print("\n--- Predictions for the first 10 TFs ---")
    for i in range(min(10, len(predictions))):
        pred_label = 'Binds' if int(predictions[i]) == 1 else 'Does not bind'
        if probabilities is not None:
            try:
                p = float(probabilities[i])
                print(f"Label {i+1}: Prediction={pred_label}, Prob={p:.4f}")
            except Exception:
                print(f"Label {i+1}: Prediction={pred_label}")
        else:
            print(f"Label {i+1}: Prediction={pred_label}")
else:
    print("No 'predictions' returned by model.inference; verify the saved model and inference API.")
